In [ ]:
## 🚀 SCRIPT CHẠY THỰC NGHIỆM TÙY CHỌN (APPENDIX D)
## Cấu trúc: CONFIG -> Setup -> Gen Data -> Train -> Zip
## Chạy trực tiếp trên Kaggle Notebook Cell

import os
import shutil
import sys

# ==================================================================================
# 1. CẤU HÌNH THỰC NGHIỆM (BẠN CHỈ CẦN SỬA Ở ĐÂY)
# ==================================================================================
CONFIG = {
    # --- THAM SỐ QUAN TRỌNG (Thay đổi theo Appendix D) ---
    "train_size": 1000,             # Số lượng mẫu: 1000, 10000, hoặc 100000
    "format": "10e-based",          # Định dạng: "10e-based", "decimal", "character", "underscore", "words"
    "epochs": 200,                  # Epochs khuyến nghị: 1k->200ep, 10k->100ep, 100k->20ep
    
    # --- CẤU HÌNH MẶC ĐỊNH (Thường giữ nguyên) ---
    "digits": 15,                   # Bài toán cộng 30 chữ số (Appendix D)
    "model_name": "t5-small",       # Model: t5-small hoặc t5-base
    "batch_size": 128,               # Batch size vật lý (Nếu OOM thì giảm xuống 8)
    "lr": 3e-4,                     # Learning rate
    "seed": 42,                     # Seed cố định để tái hiện kết quả
    "output_root": "outputs/custom_run", # Thư mục lưu kết quả
    "check_val_every_n_epoch": 10
}

# Tự động tạo tên định danh cho lần chạy này
RUN_ID = f"{CONFIG['format']}_{CONFIG['train_size']}samples_{CONFIG['epochs']}ep"
DATA_DIR = f"data/custom/{RUN_ID}"
OUTPUT_DIR = f"{CONFIG['output_root']}/{RUN_ID}"

# ==================================================================================
# HÀM HỖ TRỢ
# ==================================================================================
def run_command(command):
    """Chạy lệnh shell và in ra command để debug"""
    print(f"\n⚡ [RUNNING]: {command}")
    exit_code = os.system(command)
    if exit_code != 0:
        print(f"❌ ERROR: Command failed with code {exit_code}")
        sys.exit(1) # Dừng ngay nếu lỗi
    return True

print("=" * 80)
print(f"🚀 BẮT ĐẦU THỰC NGHIỆM: {RUN_ID}")
print(f"   - Task: Addition {CONFIG['digits']} digits")
print(f"   - Size: {CONFIG['train_size']} | Format: {CONFIG['format']}")
print(f"   - Epochs: {CONFIG['epochs']} | Batch: {CONFIG['batch_size']}")
print("=" * 80)

# ==================================================================================
# 2. SETUP MÔI TRƯỜNG & CÀI ĐẶT
# ==================================================================================
print("\n[1/5] 🛠️ Setting up environment...")

# Copy code từ /kaggle/input nếu chưa có ở /kaggle/working
if not os.path.exists("generate_data.py"):
    print("   -> Copying scripts from /kaggle/input...")
    # Thử copy từ các thư mục con hoặc thư mục gốc
    os.system("cp -r /kaggle/input/*/*.py . 2>/dev/null || cp /kaggle/input/*.py .")

# Cài đặt thư viện cần thiết
run_command("pip install -q pytorch-lightning transformers num2words")
run_command(f"mkdir -p {DATA_DIR} {OUTPUT_DIR}")
print("✅ Environment ready.")

# ==================================================================================
# 3. SINH DỮ LIỆU (GENERATE DATA)
# ==================================================================================
print("\n[2/5] 📊 Generating Datasets...")

# Chỉ sinh lại nếu chưa có file train
if not os.path.exists(f"{DATA_DIR}/train.json"):
    # Train set: Theo size cấu hình
    cmd_train = (
        f"python generate_data.py --mode single --count {CONFIG['train_size']} "
        f"--digits {CONFIG['digits']} --strategy balanced "
        f"--format {CONFIG['format']} --only_addition "
        f"--output {DATA_DIR}/train.json --seed {CONFIG['seed']}"
    )
    
    # Val set: Cố định 1000 mẫu
    cmd_val = (
        f"python generate_data.py --mode single --count 1000 "
        f"--digits {CONFIG['digits']} --strategy random "
        f"--format {CONFIG['format']} --only_addition "
        f"--output {DATA_DIR}/val.json --seed {CONFIG['seed'] + 1}"
    )
    
    # Test set: Cố định 2000 mẫu
    cmd_test = (
        f"python generate_data.py --mode single --count 2000 "
        f"--digits {CONFIG['digits']} --strategy random "
        f"--format {CONFIG['format']} --only_addition "
        f"--output {DATA_DIR}/test.json --seed {CONFIG['seed'] + 2}"
    )
    
    run_command(cmd_train)
    run_command(cmd_val)
    run_command(cmd_test)
    print("✅ Data generation completed.")
else:
    print("   -> Data files already exist. Skipping generation.")

# ==================================================================================
# 4. HUẤN LUYỆN MODEL (TRAINING)
# ==================================================================================
print(f"\n[3/5] 🏋️ Training Model ({CONFIG['epochs']} epochs)...")

# Xử lý format 'words' cần max_length dài hơn
max_len = 512 if CONFIG['format'] == "words" or CONFIG['format'] == "10e-based" else 256

train_cmd = (
    f"python train.py "
    f"--train_file {DATA_DIR}/train.json "
    f"--val_file {DATA_DIR}/val.json "
    f"--test_file {DATA_DIR}/test.json "
    f"--output_dir {OUTPUT_DIR} "
    f"--model_name_or_path {CONFIG['model_name']} "
    f"--epochs {CONFIG['epochs']} "
    f"--train_batch_size {CONFIG['batch_size']} "
    f"--val_batch_size {CONFIG['batch_size']} "
    f"--max_seq_length {max_len} "
    f"--lr {CONFIG['lr']} "
    f"--seed {CONFIG['seed']} "
    f"--check_val_every_n_epoch {CONFIG['check_val_every_n_epoch']} "
    f"--accelerator gpu --devices 1 "
)

run_command(train_cmd)
print("✅ Training completed.")

# ==================================================================================
# 5. KIỂM TRA KẾT QUẢ & NÉN FILE
# ==================================================================================
print("\n[4/5] 📈 Quick Check Results...")
res_file = f"{OUTPUT_DIR}/results.json"
if os.path.exists(res_file):
    with open(res_file, 'r') as f:
        print(f"   -> Content of results.json: \n{f.read()}")
else:
    print("⚠️ Warning: results.json not found. Training might have failed/stopped early.")

print("\n[5/5] 📦 Zipping Results...")
zip_name = f"result_{RUN_ID}.zip"
run_command(f"zip -r {zip_name} {OUTPUT_DIR}")

print("\n" + "="*80)
print(f"🎉 DONE! Vui lòng tải file '{zip_name}' về máy.")
print("="*80)